### Introduction

This jupyter file contains all the answers to the questions posed in the practical document.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
from matplotlib.ticker import MultipleLocator
from sklearn.metrics import r2_score

## a) Implementation of the Verlet algorithm

$F[x,\dot{x},t]= 2ax - 4bx^3 +F_0\cos(\omega t)$

In [ ]:
def F(x,t):
    return 2*a*x-4*b*x**3+F0*np.cos(w*t)

$v_{half} \rightarrow \qquad v(t+\frac{h}{2}) = (1-\gamma \frac{h}{2})v(t) + \frac{h}{2} F(x(t),t)\\$
$x_{new} \rightarrow \qquad x(t+h) = x(t) + hv(t+\frac{h}{2})\\$
$v_{new} \rightarrow \qquad \frac{v(t+\frac{h}{2})+\frac{h}{2}F(x(t+h), t+h)}{1+\gamma \frac{h}{2}}$

In [ ]:
def Verlet(x,v,t,h):
    v_half = (1-gamma*h/2)*v+h/2*F(x,t)
    x_new = x + h * v_half
    F_new = F(x_new,t+h)
    v_new =(v_half+h/2*F_new)/(1+h/2*gamma)
    return x_new,v_new

### Verlet Error Analysis

To show numerically that for a conservative Hamiltonian this algorithm preserves the energy we remove the $\gamma$ and $F_0$ terms by setting them equal to zero.

In [ ]:
# Defining various constants
gamma = 0
a = 0.5
b = 0.25

F0 = 0
w= 2.4

# Creating the necessary time array
N = 100000
t = np.linspace(0,50,N)
h = t[1]-t[0]

# Initial conditions
x0 = 0.5
v0 = 0


In [ ]:
# Setting up the iteration results array
x = np.zeros(N)
v = np.zeros(N)
x[0],v[0] = x0,v0

# Using Verlet algorithm
for i in range(1,N):
    x[i],v[i] = Verlet(x[i-1],v[i-1],t[i-1],h)


# Calculating The total energy
T = 0.5*v**2        # Kinetic energy (J)
V = -a*x**2+b*x**4  # Potential energy (J) from integrating -F(x,t)
H = T+V             # Hamiltonian

# Examining numerical error over time: If H is conserved it should remain te same as in the beginning
H0 = H[0]
error = H-H0


# Plotting H
plt.ylabel("H (J)", fontsize=14)
plt.xlabel("t (s)", fontsize=14)
plt.plot(t,H, 'r-')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.title("Total energy over time")
plt.ylim(H0*0.1,H0*1.5)
plt.tight_layout()
plt.show()

# Plotting code for the error
plt.ylabel(rf"$\epsilon_H$ (nJ)", fontsize=14)
plt.xlabel(r"$t$ (s)", fontsize=14)
plt.plot(t,error*1e9, 'b-', label=rf"Hamiltonian Error $\epsilon_H$")
# plt.plot(t,(V-V[0])*80) # Used to check where the shape of the error came from
# plt.plot(t,(T-T[0])*80)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12) 
plt.tight_layout()
plt.savefig("Graphs/Total error over time.png", dpi=300, bbox_inches="tight")
plt.title("Total error over time")
plt.show()


We see that the Hamiltonian stays constant, but the error seems to be oscillating over time. By finding the peaks we can examine if the error changes over time.

In [ ]:
# Using find peaks to find the top peaks of the oscillation.
peak_indices, _ = find_peaks(error, height=1e-9)
error_p = error[peak_indices]
t_p = t[peak_indices]

#Printing the difference
error_diff = error_p[-1]-error_p[0]
print('The computational error at t=%.f is %.1e J. The error oscillation increased with approximately %.1e J over time.'%(t[-1], error[-1], error_diff))

#Plotting error peaks
plt.ylabel(rf"$\epsilon_H$ (nJ)", fontsize=14)
plt.xlabel("t (s)", fontsize=14)

error_y = error_p*1e9
plt.plot(t_p,error_y, 'r.', label=rf'Hamiltonian Error Peaks')
# plt.plot(t,error*1e9)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12) 
plt.tight_layout()
plt.savefig("Graphs/Error over time.png", dpi=300, bbox_inches="tight")
plt.title("Error at peaks through time")
plt.show()

We can see that the error increases over time, but only a neglible amount.

## Verifiying the existence and dependancy on h of the shadow Hamiltonians

In [ ]:
#Create some necessary arrays
n_par = 25
h = np.geomspace(1e-2,1e-6,n_par)
error_std = np.zeros(n_par)


for i in range(n_par): # Loop over al the different h arrays
    t = np.arange(0,30,h[i])
    N = len(t)
    x = np.zeros(N)
    v = np.zeros(N)
    x[0], v[0] = 0.5,0
    for j in range(1, N):  #Solve for x and v using Verlet algorithm
        x[j], v[j] = Verlet(x[j - 1], v[j - 1], t[j - 1],h[i])
    # Calculating The total energy
    T = 0.5*v**2        # Kinetic energy (J)
    V = -a*x**2+b*x**4  # Potential energy (J) from integrating -F(x,t)
    H = T+V             # Hamiltonian

    # Examining numerical error over time
    H0 = H[0]
    error = H-H0
    error_std[i] = np.std(error)


In [ ]:
# some plotting code
plt.plot(h,error_std, label=rf'Hamiltonian Error due to $h$')
plt.yscale("log"),plt.xscale("log")
plt.ylabel(r"$\tilde{\epsilon}_H$ (J)", fontsize=14)
plt.xlabel(r"$h$ (s)", fontsize=14)
plt.xticks(fontsize=12) 
plt.yticks(fontsize=12) 
plt.legend(fontsize=12) 
plt.grid()
plt.tight_layout()
plt.savefig("Graphs/std error vs h.png", dpi=300, bbox_inches="tight")
plt.title("Standard Deviation vs Timestep")
plt.show()

In [ ]:
# Define a function to preform a lineair fit
def lin_fit(x,a,b):
    return a*x+b

val,cov = curve_fit(lin_fit,np.log(h),np.log(error_std))
u = np.sqrt(cov[0,0])
print("The dependancy on h of the error goes with the power of %.2f +/- %.2f"%(val[0],u))

The figure above shows a very strong $h^2$ dependency. Verifying the presence of the shadow Hamiltonian.

### Investigate the influence of $a$ and $b$ on the error

In [ ]:
#Define some necessary constants and arrays
n_par = 1000
h = 1e-3
b=0.25
a_array = np.logspace(-3,1,n_par) # Create the different values for a
error_std = np.zeros(n_par)
t = np.arange(0,30,h)
N = len(t)


#Solving Verlet for all values of a defined
for i in range(n_par):
    x = np.zeros(N)
    v = np.zeros(N)
    x[0], v[0] = 0.5,0
    a = a_array[i]
    for j in range(1, N):  #Solve for x and v using Verlet algorithm
        x[j], v[j] = Verlet(x[j - 1], v[j - 1], t[j - 1],h)
    # Calculating The total energy
    T = 0.5*v**2        # Kinetic energy (J)
    V = -a*x**2+b*x**4  # Potential energy (J) from integrating -F(x,t)
    H = T+V             # Hamiltonian

    # Examining the standard deviation of the error.
    H0 = H[0]
    error = H-H0
    error_std[i] = np.std(error)

In [ ]:
minimum_a = np.argmin(error_std)
print("The error mimima is at a = %.3f"%(a_array[minimum_a]))

# Some plotting code
plt.plot(a_array,error_std, label=rf'Hamiltonian Error due to $a$')
plt.yscale("log")
plt.xscale("log")
plt.ylabel(r"$\epsilon_H$ (J)", fontsize=14)
plt.xlabel(rf"$a$", fontsize=14)
plt.xticks(fontsize=12) 
plt.yticks(fontsize=12) 
plt.legend(fontsize=12) 
plt.grid()
plt.savefig("Graphs/std error vs a.png", dpi=300, bbox_inches="tight")
plt.title(r"Standard Deviation vs $a$")
plt.show()

In [ ]:
# Defining some necessary constants and arrays
n_par = 1000
h = 1e-3
b_array = np.logspace(-4,3,n_par)
a = 0.5
error_std = np.zeros(n_par)

# solving the Duffing oscillator for various diferent values of b
for i in range(n_par):
    t = np.arange(0,30,h)
    N = len(t)
    x = np.zeros(N)
    v = np.zeros(N)
    x[0], v[0] = 0.5,0
    b = b_array[i]
    for j in range(1, N):  #Solve for x and v using Verlet algorithm
        x[j], v[j] = Verlet(x[j - 1], v[j - 1], t[j - 1],h)
    # Calculating The total energy
    T = 0.5*v**2        # Kinetic energy (J)
    V = -a*x**2+b*x**4  # Potential energy (J) from integrating -F(x,t)
    H = T+V             # Hamiltonian

    # Examining the standard deviation of the numerical error
    H0 = H[0]
    error = H-H0
    error_std[i] = np.std(error)

In [ ]:
minimum_b = np.argmin(error_std)
print("The error mimima is at b = %.3f"%(b_array[minimum_b]))

# Some plotting code
plt.plot(b_array,error_std, label=rf'Hamiltonian Error due to $b$')
plt.yscale("log")
plt.xscale("log")
plt.ylabel(r"$\epsilon_H$ (J)", fontsize=14)
plt.xlabel(rf"$b$", fontsize=14)
plt.xticks(fontsize=12) 
plt.yticks(fontsize=12) 
plt.legend(fontsize=12) 
plt.grid()
plt.savefig("Graphs/std error vs b.png", dpi=300, bbox_inches="tight")
plt.title(r"Standard Deviation vs $b$")
plt.show()

In the two figures above it can be seen that a minimum occurs at $b$ = 1 and at $a$ = 0.125. These correspond to the values for what the force is zero at t=0, so in these simulations the particla doesnt move. It can be seen that for the $b$ graph the error grows the further it gets away form $b$ = 1. The reason this error grows is because for larger $b$ the potential well becomes steeper and the particle moves much faster, for lower $b$'s the particle becomes less stable and thus the error will grow. The $a$ graph can be seen to also grow for lager values than its minimum, for smaller values the error saturates because of the error due to $b$.

## b) Duffing Oscillator using Verlet Algorithm

In [ ]:
# Defining various constants
gamma = 0.1
a = 0.5
b = 0.25
F0 = 2
w= 2.4

# Creating the necessary time array
h = 1e-4
t = np.arange(0,50,h)
N =  len(t)

# Initial conditions
x0 = 0.5
v0 = 0

In [ ]:
n_par = 5 # Number of different values for x0

# x0 and v0 definen
x0_array = np.linspace(0.499,0.501,n_par)
v0_array = np.zeros([n_par])

# Plotten van x-t en x-p diagrammen
plt.figure(figsize=[14,4])
for j in range(n_par):                          #Looping over all the initial conditions
    x = np.zeros(N)
    v = np.zeros(N)
    x[0],v[0] = x0_array[j],v0_array[j]

    for i in range(1,N):                        #Solve for x and v using Verlet algorithm
        x[i],v[i] = Verlet(x[i-1],v[i-1],t[i-1],h)

    plt.subplot(121) # Plot x,t diagram left
    plt.plot(t,x,label="$x_0$ = %.4f m" %(x0_array[j]))
    plt.subplot(122) # Plot x,p diagram right
    plt.plot(x,v,label="$x_0$ = %.4f m" %(x0_array[j]))



# Setting up the right notation
plt.subplot(122)
plt.xlabel(r"$x$ (m)", fontsize=18)
plt.ylabel(r"$v$ (m/s)", fontsize=18)
plt.legend(loc="lower left",fontsize=16)
plt.xticks(fontsize=16) 
plt.yticks(fontsize=16) 
plt.grid()

# plt.xlim(-2.2,3.5)
plt.subplot(121)
plt.xlabel(r"$t$ (s)", fontsize=18)
plt.ylabel(r"$x$ (m)", fontsize=18)
# plt.xlim(15,20)

plt.xticks(fontsize=16) 
plt.yticks(fontsize=16) 
plt.grid()

plt.savefig("Graphs/x small change.png", dpi=300, bbox_inches="tight")
plt.title(r"Small change in $x$")
plt.show()

In [ ]:
n_par = 5 # number of different values for x0

# x0 and v0 definen
v0_array = np.linspace(-0.01,0.01,n_par)
x0_array = np.zeros([n_par])+0.5

# Plotten van x-t en x-p diagrammen
x[0],v[0] = x0,v0
plt.figure(figsize=[14,4])
for j in range(n_par): # Loop over all the initial conditions
    x = np.zeros(N)
    v = np.zeros(N)
    x[0],v[0] = x0_array[j],v0_array[j]
    for i in range(1,N):
        x[i],v[i] = Verlet(x[i-1],v[i-1],t[i-1],h) # Solve for x and v using Verlet algorithm
    plt.subplot(121)
    plt.plot(t,x,label="$v_0$ = %.3f m/s" %(v0_array[j]))
    plt.subplot(122)
    plt.plot(x,v,label="$v_0$ = %.3f m/s" %(v0_array[j]))

# Setting up the right notation
plt.subplot(122)
plt.xlabel(r"$x$ (m)", fontsize=18)
plt.ylabel(r"$v$ (m/s)", fontsize=18)
plt.legend(loc="lower left",fontsize=16)
plt.xticks(fontsize=16) 
plt.yticks(fontsize=16) 
plt.grid()
# plt.xlim(-2.2,3.5)

plt.subplot(121) # Plot x,p diagram right
plt.xlabel(r"$t$ (s)", fontsize=18)
plt.ylabel(r"$x$ (m)", fontsize=18)
# plt.xlim(0,40)
plt.xticks(fontsize=16) 
plt.yticks(fontsize=16) 
plt.grid()

plt.savefig("Graphs/v small change.png", dpi=300, bbox_inches="tight")
plt.title(r"Small change in $v$")
plt.show()

### Testing variance in h values

In [ ]:
n_par = 5 # number of different values for x0
h_array = np.logspace(-6,-3,n_par)

# x0 and v0 definen
x0 = 0.5
v0 = 0

# Plotten van x-t en x-p diagrammen
plt.figure(figsize=[14,4])
for j in range(n_par): # Loop over all the h values
    h = h_array[j]
    t = np.arange(0,50,h)
    N = len(t)
    x = np.zeros(N)
    v = np.zeros(N)
    x[0],v[0] = x0,v0
    for i in range(1,N):
        x[i],v[i] = Verlet(x[i-1],v[i-1],t[i-1],h) # Solve for x and v using Verlet algorithm
    plt.subplot(121)
    plt.plot(t,x,label="$h$ = %.6f m/s" %(h))
    plt.subplot(122)
    plt.plot(x,v,label="$h$ = %.6f m/s" %(h))

# Some plotting code
plt.subplot(122)
plt.xlabel(r"$x$ (m)", fontsize=18)
plt.ylabel(r"$v$ (m/s)", fontsize=18)
plt.legend(loc="lower left",fontsize=16)
plt.xticks(fontsize=16) 
plt.yticks(fontsize=16) 
plt.grid()

plt.subplot(121) # Plot x,p diagram right
plt.xlabel(r"$t$ (s)", fontsize=18)
plt.ylabel(r"$x$ (m)", fontsize=18)

plt.xticks(fontsize=16) 
plt.yticks(fontsize=16) 
plt.grid()

plt.savefig("Graphs/h chaos.png", dpi=300, bbox_inches="tight")
plt.title(r"Small change in $h$")
plt.show()

## c)  Plotting the Strange Attractor

In [ ]:
N = 250000000 
N_T = 10000
T = 2*np.pi/w
t = np.arange(0,N*T/N_T,T/N_T)
h = t[1]-t[0]
n = np.arange(0,N,N_T)

In [ ]:
# x0 and v0 definen
x0 = 0.5
v0 = 0

# Calculating Via Verlet algorithm
x = np.zeros(N)
v = np.zeros(N)
x[0],v[0] = x0,v0
for i in range(1,N):
    x[i],v[i] = Verlet(x[i-1],v[i-1],t[i-1],h) # Solve for x and v using Verlet algorithm

In [ ]:
plt.figure(figsize=(8,4))

plt.subplot(131)
plt.plot(x[n],v[n],"c.",ms=0.1)
plt.xlabel(r"$x$ (m)", fontsize=14)
plt.ylabel(r"$v$ (m/s)", fontsize=14)
plt.xticks(fontsize=12) 
plt.yticks(fontsize=12) 

plt.subplot(132)
plt.xlim(-0.5,0.5)
plt.ylim(-0.5,0.5)
plt.plot(x[n],v[n],"c.",ms=1)
plt.xlabel(r"$x$ (m)", fontsize=14)
plt.ylabel(r"$v$ (m/s)", fontsize=14)
plt.xticks(fontsize=12) 
plt.yticks(fontsize=12) 


plt.subplot(133)
plt.xlim(-0.35,-0.05)
plt.ylim(-0.35,-0.05)
plt.plot(x[n],v[n],"c.",ms=1)
plt.xlabel(r"$x$ (m)", fontsize=14)
plt.ylabel(r"$v$ (m/s)", fontsize=14)
plt.xticks(fontsize=12) 
plt.yticks(fontsize=12) 

plt.plot(t[0:int(N/10000)],x[0:int(N/10000)])

plt.tight_layout(pad=-1)
plt.savefig("Graphs/Poincare map.png", dpi=300, bbox_inches="tight")
plt.title(r"Poincaré Map")
plt.show()

## d)

When you have a line $l$ and you cover it with boxes of size $b\times b$, the amount boxes needed would be $N=\frac{l}{b}$. If we scale the box sizes with $s$, the amount of boxes needed to cover the same length would also scale with $s$ because of the self-similar geometrical property $N(b)=\frac{l}{(s\cdot b)}$. Therefore, the result $N(b)\sim b^{-1}$ implies a dimension of one.

When you have a square of length $l$ and height $h$, the amount of $b\times b$ boxes will now follow $N=\frac{l\cdot h}{b\cdot b} = \frac{l\cdot h}{b^2}$. Again, by scaling the box size with $s$, the amount of boxes needed to cover the square would be $N(b)=\frac{l}{(s\cdot b)^2}$. Therefore, the result $N\sim b^{-2}$ implies a dimnesion of two.

## e)

In [ ]:
#Normalizing to [0,1]
def norm(p):
    shifted = p-np.min(p)               #Shifting minimal p value to 0
    width = np.max(p)-np.min(p)
    scale = 1/width                     #Scaling factor to get width=1

    return shifted*scale

norm_x = norm(x[n])
norm_v = norm(v[n])


#Calculating dimension
def dimcount(l):
    #First multiplying by l and flooring to get an integer value belonging to a square
    x_cnt = np.floor(norm_x*l)
    y_cnt = np.floor(norm_v*l)

    #Boundaries check
    x_cnt[x_cnt==1] = l-1
    y_cnt[y_cnt==1] = l-1

    vec = np.vstack((x_cnt,y_cnt)).T        #Storing everything as a vector for the following
    uniques = np.unique(vec, axis=0)        #Removing duplicates to get the amount of boxes that have at least one point
    return len(uniques)


#Storing the l values
exp_max = 10
l = np.zeros([exp_max])
for i in range(0,exp_max):
    l[i] = 2**(i+1)


#Calculating N
amount = np.zeros([len(l)])
for i in range(len(l)):
    amount[i] = dimcount(l[i])


#Plotting (sanity check)
plt.plot(norm_x,norm_v,"c.",ms=1)

plt.gca().xaxis.set_minor_locator(MultipleLocator(1/16))
plt.gca().yaxis.set_minor_locator(MultipleLocator(1/16))

plt.grid(which='minor', linestyle=':', alpha=0.5)

plt.xlim(0,1)
plt.ylim(0,1)
plt.xlabel("x (m)")
plt.ylabel("v (m/s)")
plt.show()

In [ ]:
#Curve Fitting
def N_b(b,D):
    return b**(-D)

val, cov = curve_fit(N_b,1/l, amount)
x_a = np.linspace(min(1/l),max(1/l),1000)
y_a = N_b(x_a, *val)

#Storing
D = val[0]
u_D = np.sqrt(cov[0,0])

#Linear regression
amount_fit = N_b(1/l, *val)
R2 = r2_score(amount,amount_fit)

#printing results
print('The dimension of the attractor is: D =  %.2f +/- %.2f with an R2 value of %.3f. This is not an integer, and therefore it is a strange attractor'%(D, u_D, R2))

#Plotting
plt.plot(-np.log(1/l), np.log(amount), ".", label=rf'Calculated Data')
plt.plot(-np.log(x_a), np.log(y_a), "-", label=rf'Curve Fit')

plt.ylabel(r"$\log(N)$ (-)", fontsize=14)
plt.xlabel(r"$\log(b)$", fontsize=14)
plt.xticks(fontsize=12) 
plt.yticks(fontsize=12) 
plt.legend(fontsize=12) 
plt.grid()
plt.tight_layout()
plt.savefig("Graphs/fractal Fit", dpi=300, bbox_inches="tight")
plt.title("Fractal Dimension Analysis")
plt.show()